In [1]:
from wpimath.geometry import Pose3d as wpiPose3d, Translation3d as wpiTranslation3d, Rotation2d as wpiRotation2d, Translation2d as wpiTranslation2d, Quaternion as wpiQuaternion, Rotation3d as wpiRotation3d
import math

In [2]:
shooter_pose = wpiPose3d()
target_point = wpiTranslation3d()

In [3]:
def normalizeAngleRadians(angle: float) -> float:
    angle_normalized = angle % math.tau
    if angle_normalized >= math.pi:
        angle_normalized -= math.tau
    return angle_normalized


def computeAngleDifferenceRadians(angle1: float, angle2: float) -> float:
    return normalizeAngleRadians(angle1 - angle2)


def computeRobotRotationToAlign(
    shooter_pose3d: wpiPose3d,
    target: wpiTranslation3d,
) -> wpiRotation2d:
    shooter_to_target = (target - shooter_pose3d.translation()).toTranslation2d()
    shooter_to_target_angle = shooter_to_target.angle().radians()
    shooter_angle = shooter_pose3d.rotation().z
    angle_rad = computeAngleDifferenceRadians(shooter_to_target_angle, shooter_angle)
    return wpiRotation2d(angle_rad)

In [4]:
%%timeit
_ = computeRobotRotationToAlign(shooter_pose, target_point)

955 μs ± 116 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [14]:
pi_over_two = math.pi / 2
pi = math.pi
two_pi = 2 * math.pi

def normalize_angle_degrees(degrees):
    return (degrees + 180) % 360 - 180

def normalize_angle_rad(rad):
    return (rad + pi) % two_pi - pi

class Rotation2d:
    @classmethod
    def fromDegrees(cls, degrees):
        return cls(math.radians(degrees))

    @classmethod
    def fromRadians(cls, radians):
        return cls(radians)

    def __init__(self, rad):
        self._radians = rad

    def radians(self):
        return self._radians

    def degrees(self):
        return math.degrees(self._radians)

    def __add__(self, other: Rotation2d):
        return Rotation2d(normalize_angle_rad(self._radians + other._radians))

    def __sub__(self, other: Rotation2d):
        return Rotation2d(normalize_angle_rad(self._radians - other._radians))

    def cos(self):
        return math.cos(self._radians)

class Translation2d:
    @classmethod
    def fromWPI(cls, wpi_translation: wpiTranslation2d):
        return cls(wpi_translation.x, wpi_translation.y)

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def angle(self):
        return Rotation2d(math.atan2(self.y, self.x))

    def toWPI(self):
        return wpiTranslation2d(self.x, self.y)

class Translation3d:
    @classmethod
    def fromWPI(cls, wpi_translation: wpiTranslation3d):
        return cls(wpi_translation.x, wpi_translation.y, wpi_translation.z)

    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

    def __sub__(self, other: Translation3d) -> Translation3d:
        return Translation3d(self.x - other.x, self.y - other.y, self.z - other.z)

    def toWPI(self):
        return wpiTranslation3d(self.x, self.y, self.z)

    def toTranslation2d(self):
        return Translation2d(self.x, self.y)

class Quaternion:
    @classmethod
    def fromWPI(cls, wpi_quaternion: wpiQuaternion):
        return cls(wpi_quaternion.W(), wpi_quaternion.X(), wpi_quaternion.Y(), wpi_quaternion.Z())

    def __init__(self, w, x, y, z):
        self.w = w
        self.x = x
        self.y = y
        self.z = z

    def toWPI(self) -> wpiQuaternion:
        return wpiQuaternion(self.w, self.x, self.y, self.z)


class Rotation3d:
    @classmethod
    def fromWPI(cls, wpi_rotation: wpiRotation3d):
        return cls(Quaternion.fromWPI(wpi_rotation.getQuaternion()))

    def __init__(self, quaternion: Quaternion):
        self.q = quaternion

    @property
    def z(self):
        w = self.q.w
        x = self.q.x
        y = self.q.y
        z = self.q.z

        cycz = 1.0 - 2.0 * (y * y + z * z)
        cysz = 2.0 * (w * z + x * y)
        cy_sq = cycz * cycz + cysz * cysz
        if cy_sq > 1e-20:
          return math.atan2(cysz, cycz)
        else:
          return math.atan2(2.0 * w * z, w * w - z * z)



class Pose3d:
    @classmethod
    def fromWPI(cls, wpi_pose: wpiPose3d):
        return cls(Translation3d.fromWPI(wpi_pose.translation()), Rotation3d.fromWPI(wpi_pose.rotation()))

    def __init__(self, translation: Translation3d, rotation: Rotation3d):
        self._translation = translation
        self._rotation = rotation

    def translation(self):
        return self._translation

    def rotation(self):
        return self._rotation


def computeRobotRotationToAlign(
    shooter_pose3d: Pose3d,
    target: Translation3d,
) -> wpiRotation2d:
    shooter_to_target = (target - shooter_pose3d.translation()).toTranslation2d()
    shooter_to_target_angle = shooter_to_target.angle().radians()
    shooter_angle = shooter_pose3d.rotation().z
    angle_rad = computeAngleDifferenceRadians(shooter_to_target_angle, shooter_angle)
    return wpiRotation2d(angle_rad)


In [15]:
%%timeit
computeRobotRotationToAlign(Pose3d.fromWPI(shooter_pose), Translation3d.fromWPI(target_point))

14.8 μs ± 342 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
